# 3dvid on Kaggle or Colab

Runs `stylize.py` (voxel / contour intro) and iw3 (half-SBS 3D) on a free GPU. The same notebook works on both; it detects which one it is on.

**Kaggle** (better for full renders: 4 CPUs, background runs)
1. Once per account: verify your phone number in Kaggle *Settings* (GPU and Internet stay greyed out until you do).
2. Right panel, *Session options*: **Accelerator = GPU T4 x2** (not P100: recent torch builds dropped its architecture) and **Internet = On**.
3. *+ Add Input*, *Upload*, *New Dataset*: drop your clip in. It appears under `/kaggle/input/`.
4. *Run All*. For a full render use *Save Version*, *Save & Run All (Commit)*: it runs in the background (up to 12 h), you can close the tab, and the files land in the version's *Output* tab.

**Colab** (fine for `--frame` / `--preview`; only 2 CPUs and no background runs, so a full clip means keeping the tab open for hours)
1. *Runtime*, *Change runtime type*: **T4 GPU**.
2. *File*, *Save a copy in Drive*, or your edits are lost.
3. Upload the clip with the folder icon on the left (it lands in `/content`), or put it in `MyDrive/3dvid/` and mount Drive.
4. *Run All*.

**Two ways to use it.** The cells below render one clip from the command line. The last section instead starts the **studio** (web UI + API) on a public URL with this GPU behind it, so you can use it from a phone or laptop like a server, as long as this notebook stays open.

The GPU only speeds up the depth + mask models; the effects run on the CPUs. Iterate with `--frame` / `--preview`. The working folder is wiped when the session ends, so download what you want to keep.

In [ ]:
# --- settings -----------------------------------------------------------------
REPO_REF = "main"   # branch of github.com/mwihoti/3dvid to run
CLIP = None          # None = first video found under /kaggle/input, or a full path

# stylize.py flags. Fastest first:  --frame 4  (one PNG at 4 s)  ->  --preview 3  ->  no limit (full clip)
ARGS = "--preview 3 --effect voxel --bg plasma --layout wide"

MAKE_3D = False      # also render a half-SBS 3D video with iw3 (same flags as convert3d.sh)

In [ ]:
# --- setup: idempotent, re-running only pulls the latest code -----------------
import os, shutil, urllib.request
ON_KAGGLE = os.path.isdir("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
GPU_HINT = ("right panel -> Session options -> Accelerator -> GPU T4 x2" if ON_KAGGLE
            else "Runtime -> Change runtime type -> T4 GPU")
NET_HINT = ("right panel -> Session options -> Internet -> On (needs a phone-verified Kaggle account)"
            if ON_KAGGLE else "check your connection")
ALLOW_CPU = False   # True = carry on without a GPU (depth + mask then take many minutes)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab")
%cd {WORK}

# stop at the first missing setting, instead of a cascade of follow-on errors
if not shutil.which("nvidia-smi") and not ALLOW_CPU:
    raise RuntimeError(f"No GPU in this session. Turn it on: {GPU_HINT}. The session restarts; then Run All again.")
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise RuntimeError(f"No internet in this session, so the code can't be downloaded. Turn it on: {NET_HINT}. Then Run All again.")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null)
!test -d 3dvid || git clone -q https://github.com/mwihoti/3dvid.git
!git -C 3dvid fetch -q origin && git -C 3dvid checkout -q -B {REPO_REF} origin/{REPO_REF}
# nunif (iw3) at the commit pinned in .gitmodules
!git -C 3dvid submodule update --init -q
if not os.path.isdir("3dvid/nunif/iw3"):
    raise RuntimeError("Downloading the code failed (see the git messages above). Re-run this cell.")
# torch, numpy, opencv and scikit-learn come preinstalled with CUDA; don't reinstall them.
# rembg[gpu] brings onnxruntime-gpu, which must not sit next to the CPU onnxruntime.
!pip uninstall -y -q onnxruntime 2>/dev/null; pip install -q -r 3dvid/nunif/requirements.txt "rembg[gpu]>=2.0.60"

In [ ]:
# --- sanity check -------------------------------------------------------------
import torch, onnxruntime as ort
if torch.cuda.is_available():
    print("torch", torch.__version__, "on", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    if torch.cuda.get_device_capability(0) < (7, 0):
        print("WARNING: this GPU is too old for current torch; switch to a T4: " + GPU_HINT)
else:
    print("WARNING: torch sees no GPU, depth will run on CPU (slow). " + GPU_HINT)
# CUDAExecutionProvider here means the person mask runs on the GPU too; if it is missing
# rembg silently falls back to CPU, which still works but makes the mask pass ~10x slower.
print("onnxruntime", ort.__version__, ort.get_available_providers())

In [ ]:
# --- pick the clip --------------------------------------------------------------
import glob, re
VIDEO_EXT = (".mp4", ".mov", ".m4v", ".mkv", ".webm", ".avi")
OUT_DIR = os.path.join(WORK, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
if CLIP is None:
    if ON_KAGGLE:
        candidates = glob.glob("/kaggle/input/**/*", recursive=True)
    else:
        # top level of /content (the Files panel upload) + MyDrive/3dvid if Drive is mounted;
        # never the outputs folder, or a rerun would pick up its own render
        candidates = glob.glob("/content/*") + glob.glob("/content/drive/MyDrive/3dvid/**/*", recursive=True)
    found = sorted(p for p in candidates if p.lower().endswith(VIDEO_EXT))
    where = "+ Add Input -> Upload -> New Dataset" if ON_KAGGLE else "the folder icon on the left -> upload (or MyDrive/3dvid/)"
    assert found, f"No video found: upload your clip via {where}, then rerun this cell"
    CLIP = found[0]
    if len(found) > 1:
        print("other clips (set CLIP to use one):", *found[1:], sep="\n  ")
print("clip:", CLIP)

# depth + mask cache: re-renders of the same clip in this session skip both models
os.environ["STYLIZE_CACHE_DIR"] = os.path.join(WORK, "cache")
name = os.path.splitext(os.path.basename(CLIP))[0]
# name outputs after the flags so different settings don't overwrite each other
OUT_BASE = os.path.join(OUT_DIR, name + "_" + re.sub(r"[^a-z0-9]+", "_", ARGS.lower()).strip("_")[:60])

In [ ]:
# --- render -----------------------------------------------------------------------
!cd {WORK}/3dvid && python stylize.py "{CLIP}" {ARGS} -o "{OUT_BASE}"

In [ ]:
# --- optional: half-SBS 3D for a VR headset (mirrors convert3d.sh, but on the GPU) ---
if MAKE_3D:
    out3d = os.path.join(OUT_DIR, name + "_3d.mp4")
    get_ipython().system(f'cd {WORK}/3dvid/nunif && python -m iw3.cli --gpu 0 --depth-model Any_V2_S '
                         f'--divergence 2.0 --convergence 0.5 --half-sbs --max-output-height 720 '
                         f'-i "{CLIP}" -o "{out3d}" --yes')

In [ ]:
# --- show the results -----------------------------------------------------------------
from IPython.display import Video, Image, display
files = sorted(glob.glob(OUT_DIR + "/*"), key=os.path.getmtime)
for f in files:
    print(f"{os.path.getsize(f) / 1e6:7.1f} MB  {f}")
newest = [f for f in files if f.startswith(OUT_BASE)]
for f in newest:
    if f.endswith(".png"):
        display(Image(f, width=480))
    elif f.endswith(".mp4"):
        display(Video(f, embed=True, width=480))
print("Download: Output panel on the right (or the version's Output tab after a commit run)" if ON_KAGGLE
      else "Download: folder icon on the left -> outputs -> right-click -> Download")

## Studio server: use the GPU from the web UI

Instead of rendering one clip here, start `serve.py` behind a Cloudflare quick tunnel. You get a public `https://….trycloudflare.com` link to the full studio (upload, options, progress, playback), with this T4 doing depth and mask. The link also works from the camera page on GitHub Pages ("from Pages" link), so "Render exact version in studio" works there too.

Run the **settings** and **setup** cells first, then the cell below. The studio lives while this notebook is open (12 h at most); the URL and token change every session. Download finished renders from the studio page, or from `studio/outputs` in the Output panel, before the session ends.

In [ ]:
# --- studio server: the web UI + API on a public URL, this GPU behind it ----------
# Run the setup cell first. Keep this notebook open: the studio lives only while
# the session does (12 h at most), and the URL changes every time.
import os, secrets, subprocess, time, urllib.request
TOKEN = os.environ.get("AUTH_TOKEN") or secrets.token_hex(12)   # new each session unless you set one
PAGES = "https://mwihoti.github.io"   # the camera page on GitHub Pages may call this studio (CORS)
DATA = f"{WORK}/studio"               # uploads, outputs, cache - download what you want to keep
!pip install -q fastapi uvicorn python-multipart 2>&1 | grep -v -i "requires\|resolver\|conflicts"
!test -x /usr/local/bin/cloudflared || (curl -sSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared)
!pkill -f "python serve.py" 2>/dev/null; pkill -x cloudflared 2>/dev/null; sleep 1   # re-running replaces the old pair
os.makedirs(DATA, exist_ok=True)
env = dict(os.environ, AUTH_TOKEN=TOKEN, DATA_DIR=DATA, HOST="127.0.0.1", PORT="8000",
           CORS_ORIGINS=PAGES, CLOUDFLARED="/usr/local/bin/cloudflared", PAGES_URL=PAGES + "/3dvid")
server = subprocess.Popen(["python", "serve.py"], cwd=f"{WORK}/3dvid", env=env,
                          stdout=open(f"{WORK}/serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"serve.py did not start - see {WORK}/serve.log")
tunnel = subprocess.run(["bash", "deploy/tunnel.sh"], cwd=f"{WORK}/3dvid", env=env, capture_output=True, text=True)
print(tunnel.stdout, tunnel.stderr)
if tunnel.returncode:
    raise RuntimeError("the tunnel did not come up (Cloudflare blocked, or no internet) - see 3dvid/tunnel.log")
print("Open the 'studio' link above. Anyone with the URL still needs the token, so keep both private.")

In [ ]:
# --- studio: status, recent log, and stop ------------------------------------------
STOP = False    # True = stop the studio and the tunnel (the session keeps running)
import json, re, subprocess, urllib.request
if STOP:
    subprocess.run("pkill -f 'python serve.py'; pkill -x cloudflared", shell=True)
    print("stopped")
else:
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/api/jobs", headers={"Authorization": "Bearer " + TOKEN})
        for j in json.load(urllib.request.urlopen(req, timeout=3))[:5]:
            print(f"{j['status']:9} {j['label']:30} {j['stage'] or '':8} {j['percent']:5.1f}%  {j['message'][:70]}")
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{WORK}/3dvid/tunnel.log").read())
        print("public URL:", m.group(0) if m else "(none - re-run the cell above)")
    except Exception as e:
        print("studio not running:", e)
    print("--- serve.log (last 15 lines) ---")
    print(subprocess.run(["tail", "-n", "15", f"{WORK}/serve.log"], capture_output=True, text=True).stdout)